# End-to-End Production Pipeline (Streamfix v3 — Fast JIT & Q1 Metrics): Vanilla MATD3 / MASAC on Kaggle T4x2 Public Account 1
1. Bootstrap repo + install pinned dependencies (`warp-lang==1.18.0`, `simpy`, `trimesh`, `shapely`, `mapbox-earcut`, `pydantic`, `wandb`, `huggingface_hub`) + zero-leak Kaggle Secrets.
2. T4x2 GPU Acceptance Suite (`test_gpu_warp_parity_and_cudagraph_t4.py`) — measured v3 initial CUDA step + JIT 59.14s; see actual smoke log.
3. **1M-Step Real Pilot** (`--total-steps 1000000 --eval-episodes 20`) verifying real target discovery, multi-hop GCS delivery, non-null AoI/energy efficiency & gradient convergence (runtime unverified until pilot).
4. **20M-Step Resumable Production Run** (`--mode production --max-wall-clock-s 37800`) with automatic pre-eval crash recovery, W&B Online & incremental HF backups.
5. 5-Seed Paper Aggregation (`aggregate_five_seed_rliable.py`) & Q1 Forensic Summary.

In [ ]:
# Cell 1: Bootstrap Repo, Install Pinned Dependencies, Load Kaggle Secrets Safely
import os, pathlib, subprocess, sys

cwd = pathlib.Path.cwd()
if not (cwd / "uav_network").exists():
    for cand in sorted(pathlib.Path("/kaggle/input").glob("**/uav_network")):
        os.chdir(cand.parent)
        break
print("Working repo:", pathlib.Path.cwd())
os.environ["PYTHONPATH"] = str(pathlib.Path.cwd())
os.environ.setdefault("WARP_CACHE_PATH", "/kaggle/working/.warp_cache")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "warp-lang==1.18.0", "simpy>=4.0,<5.0", "trimesh>=4.12,<5",
    "shapely>=2,<3", "mapbox-earcut>=2,<3", "pydantic>=2",
    "wandb", "huggingface_hub", "pyyaml"
], check=True)

try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    for _k in ("WANDB_API_KEY", "HF_TOKEN", "HF_DATASET_REPO"):
        if not os.environ.get(_k):
            try:
                os.environ[_k] = _sec.get_secret(_k)
            except Exception:
                pass
except ImportError:
    pass

import torch, warp as wp, simpy, trimesh
wp.config.kernel_cache_dir = os.environ["WARP_CACHE_PATH"]
wp.init()
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)")
print("Warp:", wp.__version__, "| SimPy:", simpy.__version__, "| trimesh:", trimesh.__version__)
print("WANDB_API_KEY set:", bool(os.environ.get("WANDB_API_KEY")), "| HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# Cell 2: T4x2 GPU Acceptance Suite (Cold JIT ~15-25s, Parity, Streamfix, 1M VRAM, Real UavNetSim)
import subprocess, sys
subprocess.run([sys.executable, "tests/test_gpu_warp_parity_and_cudagraph_t4.py"], check=True)

In [ ]:
# Cell 3: 1M-Step Real GPU Pilot (Verifies Real Discovery, Multi-Hop Delivery, Gradient Updates & Non-Null Q1 Metrics)
import json, os, pathlib, shutil, subprocess, sys

PILOT_ALGO = os.environ.get("UAV_ALGO", "matd3")
pilot_dir = pathlib.Path(f"/kaggle/working/pilot_1m_{PILOT_ALGO}_s44")
if pilot_dir.exists():
    shutil.rmtree(pilot_dir)

subprocess.run([
    sys.executable, "-m", "uav_network.experiments.train_slotted_vanilla",
    "--config", f"configs/vanilla_{PILOT_ALGO}_shared_2d_ds_aoi.yaml",
    "--output", str(pilot_dir),
    "--mode", "production",
    "--total-steps", "1000000",
    "--eval-episodes", "20",
    "--seed", "44",
    "--wandb-mode", "online" if os.environ.get("WANDB_API_KEY") else "offline",
], check=True)

rep = json.loads((pilot_dir / "summary_metrics.json").read_text(encoding="utf-8"))
print("PILOT 1M SUMMARY (Slotted):", json.dumps({k: rep["test_slotted"].get(k) for k in ("mean_return", "target_discovery_rate_pct", "target_delivery_ratio_pct", "multihop_delivery_ratio_pct", "mean_aoi_s", "energy_kj_per_delivered_target")}, indent=2))

In [ ]:
# Cell 4: Full 20M-Transition Production Run (Resumable, Pre-Eval Crash Safe, Kaggle 12h Guard)
import os, pathlib, shutil, subprocess, sys

ALGO = os.environ.get("UAV_ALGO", "matd3")
SEED = int(os.environ.get("UAV_SEED", "44"))
MAX_WALL_S = os.environ.get("UAV_MAX_WALL_S", "37800")
HF_REPO = os.environ.get("HF_DATASET_REPO", "")
WANDB_MODE = "online" if os.environ.get("WANDB_API_KEY") else "offline"

cfg_path = f"configs/vanilla_{ALGO}_shared_2d_ds_aoi.yaml"
out_dir = pathlib.Path(f"/kaggle/working/prod_{ALGO}_s{SEED}")
out_dir.mkdir(parents=True, exist_ok=True)

RESUME_RUN_SUBPATH = os.environ.get("HF_RESUME_SUBPATH", "")
if HF_REPO and os.environ.get("HF_TOKEN") and RESUME_RUN_SUBPATH:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=HF_REPO, repo_type="dataset", token=os.environ["HF_TOKEN"],
        allow_patterns=f"{RESUME_RUN_SUBPATH}/*", local_dir=str(out_dir.parent / "hf_pull"),
    )
    pulled = out_dir.parent / "hf_pull" / RESUME_RUN_SUBPATH
    if pulled.exists():
        shutil.copytree(pulled, out_dir, dirs_exist_ok=True)

cmd = [
    sys.executable, "-m", "uav_network.experiments.train_slotted_vanilla",
    "--config", cfg_path,
    "--output", str(out_dir),
    "--mode", "production",
    "--seed", str(SEED),
    "--max-wall-clock-s", str(MAX_WALL_S),
    "--wandb-mode", WANDB_MODE,
    "--wandb-project", "uav_search_target",
]
if (out_dir / "checkpoints" / "latest_checkpoint.pt").exists():
    cmd.append("--resume")
elif any(out_dir.iterdir()):
    shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
if HF_REPO:
    cmd.extend(["--hf-repo", HF_REPO])
print("Launching:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Cell 5: 5-Seed Paper Aggregation (rliable / marl-eval)
import pathlib, subprocess, sys

completed_runs = [str(p) for p in sorted(pathlib.Path("/kaggle/working").glob("prod_*_s*")) if (p / "marl_eval_results.json").exists()]
if completed_runs:
    agg_out = "/kaggle/working/five_seed_rliable_summary.json"
    subprocess.run([
        sys.executable, "-m", "uav_network.experiments.aggregate_five_seed_rliable",
        "--runs", *completed_runs, "--out", agg_out,
    ], check=True)
    print(pathlib.Path(agg_out).read_text(encoding="utf-8"))
else:
    print("No completed production runs found yet under /kaggle/working/prod_*_s*.")

In [ ]:
# Cell 6: Inspect Final Q1 Dual-Fidelity Report (Surfaces Delivery, AoI, Energy Efficiency & Sim-to-Net Gap)
import json, pathlib
for p in sorted(pathlib.Path("/kaggle/working").glob("*_s*/summary_metrics.json")):
    rep = json.loads(p.read_text(encoding="utf-8"))
    print("===", p.parent.name, "===")
    print("gpu_warp_verified:", rep.get("gpu_warp_verified"), "| vendor_uavnetsim_verified:", rep.get("vendor_uavnetsim_verified"))
    for sec in ("test_slotted", "test_full_uavnetsim"):
        if sec in rep:
            s = rep[sec]
            print(f"  {sec}: deliv={s.get('total_targets_delivered')} ({s.get('target_delivery_ratio_pct')}%), AoI={s.get('mean_aoi_s')}s, kJ/deliv={s.get('energy_kj_per_delivered_target')}, return_iqm={s.get('return_iqm')}")
    print("  sim_to_net_gap:", json.dumps(rep.get("sim_to_net_gap", {}), indent=2))